In [1]:
# Imports
import sys
from pathlib import Path

# Resolve project root and ensure it's on sys.path
ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "pyproject.toml").exists() or (ROOT / "raw_data").exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import discretize_preprocess

In [2]:
# Preprocess data
from pathlib import Path

dataset_path = ROOT / "raw_data" / "car.csv"
output_path = ROOT / "discretized_data" / "car.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path), bins=10, strategy='uniform')

Preprocessing: /home/adity/github/katabatic-mentorship-repo/raw_data/car.csv
Saved preprocessed discrete dataset to: /home/adity/github/katabatic-mentorship-repo/discretized_data/car.csv


In [3]:
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.privbayes_alex.adapter import PrivBayes

# Set paths
input_csv = str(output_path)
output_dir = str(ROOT / "sample_data" / "car")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "car" / "privbayes")

#PrivBayes parameters
model_config = {
            "epsilon": 1.0,  # Privacy budget (lower is more private)
            "degree_of_bayesian_network": 2, # 'k' parameter in PrivBayes paper
        }
pipeline = TrainTestSplitPipeline(model=PrivBayes)

pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
    **model_config
)

Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
Loading PrivBayes data from: /home/adity/github/katabatic-mentorship-repo/sample_data/car
Training PrivBayes (epsilon=1.0, k=2) on 1382 rows...
================ Constructing Bayesian Network (BN) ================
Adding ROOT col_6
Adding attribute col_3
Adding attribute col_5
Adding attribute col_2
Adding attribute col_1
Adding attribute col_0
Adding attribute col_4
========================== BN constructed ==========================
PrivBayes description saved to /home/adity/github/katabatic-mentorship-repo/synthetic/car/privbayes/description.json
Generating synthetic data to

/home/adity/github/katabatic-mentorship-repo/katabatic/models/privbayes_alex/adapter.py:69: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')
/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [14:36:33] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/car/privbayes_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6908
F1 Score: 0.5900

MLP:
Accuracy: 0.7081
F1 Score: 0.6950

RF:
Accuracy: 0.6561
F1 Score: 0.6554

XGBoost:
Accuracy: 0.6416
F1 Score: 0.6509


'Train test split pipeline executed successfully.'